# MDA-MB-231 Hoechst Ploidy Results Analysis

- Use environment.yml

# Imports

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib import rcParams
import pathlib
from matplotlib.ticker import FuncFormatter
from matplotlib.ticker import FuncFormatter, MultipleLocator

# Functions

In [ ]:
def stats_and_plot(dataframe, alpha=0.05, xlabel=None, ylabel=None, title=None, 
                    sig_bars=True, point_label=False, fig_size=(10, 6), font_sz=18, 
                    save_svg=False, ploidy_axis=False, chromosomes_per_n=23,
                    correction='bonferroni'):
    """
    Perform pairwise statistical tests between each column of the DataFrame and 
    create a box and whisker plot with significance indicators.

    For each pair of columns, automatically detects whether the data are paired 
    (same row index/subject present in both columns, e.g. same clones measured at 
    two timepoints) or independent, and selects the appropriate test:
        Paired:      Paired t-test (if differences are normal) or Wilcoxon signed-rank
        Independent: t-test / Welch's t-test / Mann-Whitney U, based on normality 
                     and variance checks

    All pairwise raw p-values are then corrected together using either FDR 
    (Benjamini-Hochberg) or Bonferroni.

    Parameters:
    dataframe (pd.DataFrame): Input data, one column per group/condition. Row index 
        should identify the subject/clone so paired comparisons can be detected.
    alpha (float): Significance threshold. Default 0.05.
    correction (str): 'fdr' (Benjamini-Hochberg, default) or 'bonferroni'.
    ploidy_axis (bool): If True, format y-axis ticks as "1N", "2N", etc.
    chromosomes_per_n (float): Value representing 1N in the data's units 
        (e.g. 23 for raw chromosome counts, 1 if data is already in N units).
    save_svg (bool): If True, saves the plot as an SVG using the title as filename.

    Returns:
    pd.DataFrame: Symmetric matrix of corrected (adjusted) p-values between each 
        pair of columns.
    """
    import matplotlib as mpl
    from itertools import combinations
    from scipy.stats import (shapiro, levene, mannwhitneyu, ttest_ind, 
                              ttest_rel, wilcoxon)
    from statsmodels.stats.multitest import multipletests
    from matplotlib.ticker import FuncFormatter, MultipleLocator

    assert correction in ('fdr', 'bonferroni'), "correction must be 'fdr' or 'bonferroni'"

    dataframe = dataframe.apply(pd.to_numeric, errors='coerce')
    columns = list(dataframe.columns)
    num_samples = len(columns)

    fig, ax = plt.subplots(figsize=fig_size)

    # --- Clean spine styling ---
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.yaxis.set_ticks_position('left')
    ax.xaxis.set_ticks_position('bottom')

    # --- Boxplot ---
    bp = ax.boxplot(
        [dataframe[col].dropna() for col in columns],
        labels=columns,
        showmeans=True,
        showfliers=False,
        patch_artist=True,
        meanprops=dict(marker='D', markerfacecolor='black', markeredgecolor='black', markersize=5),
        medianprops=dict(color='black', linewidth=2),
        boxprops=dict(facecolor='white', color='black', linewidth=1.5),
        whiskerprops=dict(color='black', linewidth=1.5, linestyle='--'),
        capprops=dict(color='black', linewidth=1.5),
    )

    # --- Jittered data points ---
    for i, col in enumerate(columns, start=1):
        y = dataframe[col].dropna()
        x = np.random.normal(i, 0.04, len(y))
        ax.scatter(x, y, color='dimgray', alpha=0.6, s=30, zorder=3, edgecolors='none')

        if point_label:
            valid_indices = dataframe[col].dropna().index
            for x_val, y_val, label in zip(x, y, valid_indices):
                ax.text(x_val, y_val, str(label), fontsize=8, ha='right', va='bottom')

    # --- Overall assumption testing (reported for reference; per-pair checks done below) ---
    alpha_assumption = 0.05
    groups = [dataframe[col].dropna().values for col in columns]
    normality_ok = all(shapiro(g).pvalue > alpha_assumption for g in groups if len(g) >= 3)
    levene_p = levene(*groups).pvalue if len(groups) > 1 else 1.0
    variance_ok = levene_p > alpha_assumption

    print(f"Overall normality (Shapiro-Wilk):     {'PASS' if normality_ok else 'FAIL'}")
    print(f"Overall equal variance (Levene's):    {'PASS' if variance_ok else 'FAIL'}")
    print(f"Correction method:                    {'Benjamini-Hochberg (FDR)' if correction == 'fdr' else 'Bonferroni'}")
    print()

    # --- Pairwise comparisons: adaptively choose paired vs. independent test per pair ---
    pair_results = []

    for col1, col2 in combinations(columns, 2):
        s1, s2 = dataframe[col1], dataframe[col2]
        idx1, idx2 = s1.dropna().index, s2.dropna().index
        shared_idx = idx1.intersection(idx2)

        is_paired = (len(shared_idx) >= 2) and (set(idx1) == set(idx2))

        if is_paired:
            v1 = s1.loc[shared_idx].values
            v2 = s2.loc[shared_idx].values
            diffs = v1 - v2

            diff_normal = shapiro(diffs).pvalue > alpha_assumption if len(diffs) >= 3 else False

            if diff_normal:
                test_used = "Paired t-test"
                _, p_raw = ttest_rel(v1, v2)
            else:
                test_used = "Wilcoxon signed-rank"
                try:
                    _, p_raw = wilcoxon(v1, v2)
                except ValueError:
                    test_used = "Wilcoxon signed-rank (degenerate, no variance)"
                    p_raw = 1.0
        else:
            g1, g2 = s1.dropna().values, s2.dropna().values
            if len(g1) < 2 or len(g2) < 2:
                continue  # not enough data to compare

            n1_ok = shapiro(g1).pvalue > alpha_assumption if len(g1) >= 3 else False
            n2_ok = shapiro(g2).pvalue > alpha_assumption if len(g2) >= 3 else False
            pair_normal = n1_ok and n2_ok
            pair_var_ok = levene(g1, g2).pvalue > alpha_assumption

            if pair_normal and pair_var_ok:
                test_used = "Independent t-test"
                _, p_raw = ttest_ind(g1, g2, equal_var=True)
            elif pair_normal and not pair_var_ok:
                test_used = "Welch's t-test"
                _, p_raw = ttest_ind(g1, g2, equal_var=False)
            else:
                test_used = "Mann-Whitney U"
                _, p_raw = mannwhitneyu(g1, g2, alternative='two-sided')

        pair_results.append({'col1': col1, 'col2': col2, 'p_raw': p_raw,
                              'test': test_used, 'paired': is_paired})

    # --- Apply multiple comparison correction across ALL pairs together ---
    p_values_df = pd.DataFrame(np.nan, columns=columns, index=columns)

    if pair_results:
        raw_pvals = [r['p_raw'] for r in pair_results]
        method = 'fdr_bh' if correction == 'fdr' else 'bonferroni'
        _, p_adj, _, _ = multipletests(raw_pvals, alpha=alpha, method=method)

        for r, p_corrected in zip(pair_results, p_adj):
            r['p_adj'] = p_corrected
            p_values_df.loc[r['col1'], r['col2']] = p_corrected
            p_values_df.loc[r['col2'], r['col1']] = p_corrected

            tag = "PAIRED" if r['paired'] else "unpaired"
            sig = " *" if p_corrected < alpha else ""
            print(f"{r['col1']} vs {r['col2']:<20s} [{tag:8s}] {r['test']:<28s} "
                  f"raw p={r['p_raw']:.4f}  adj p={p_corrected:.4f}{sig}")

    # --- Significance bars ---
    if sig_bars and pair_results:
        y_max = dataframe.max().max()
        y_min_ax, y_max_ax = ax.get_ylim()
        y_span = y_max_ax - y_min_ax
        line_offset = y_span * 0.03
        text_offset = y_span * 0.01
        highest_y = y_max

        for r in pair_results:
            p_adj = r['p_adj']
            if p_adj < alpha:
                if p_adj < 0.001:
                    sig_symbol = '***'
                elif p_adj < 0.01:
                    sig_symbol = '**'
                else:
                    sig_symbol = '*'

                x1 = columns.index(r['col1']) + 1
                x2 = columns.index(r['col2']) + 1
                y = y_max + line_offset

                ax.plot([x1, x2], [y, y], lw=1.5, color='black')
                ax.text((x1 + x2) * 0.5, y + text_offset, sig_symbol,
                        ha='center', va='bottom', color='black', fontsize=13)

                line_offset += y_span * 0.09
                highest_y = y + text_offset

        ax.set_ylim(bottom=dataframe.min().min() * 0.95, top=highest_y * 1.02)

    # --- Ploidy-style y-axis labels (integer N only, e.g. "2N", "3N", "4N") ---
    if ploidy_axis:
        def ploidy_formatter(x, pos):
            n_value = round(x / chromosomes_per_n)
            return f"{n_value}N"
        ax.yaxis.set_major_locator(MultipleLocator(chromosomes_per_n))
        ax.yaxis.set_major_formatter(FuncFormatter(ploidy_formatter))

    # --- Labels & title ---
    ax.set_xlabel(xlabel if xlabel else 'Samples', labelpad=10)
    ax.set_ylabel(ylabel if ylabel else 'Growth Rate', labelpad=10)
    plot_title = title if title else 'Boxplot with Statistical Significance Indicators'
    ax.set_title(plot_title, pad=12, fontweight='bold')

    fig.tight_layout()

    if save_svg:
        filename = f"{plot_title}.svg"
        fig.savefig(filename, format='svg', bbox_inches='tight', dpi=300)

    plt.show()
    return p_values_df

In [ ]:
def build_ploidy_summary_231(ploidy_initial, ploidy_week5):
    records = {}

    for sample, val in ploidy_initial.items():
        if sample.startswith('C'):
            records.setdefault(sample, {})['Control Initial'] = val
        elif sample.startswith('F'):
            records.setdefault(sample, {})['Fusion Initial'] = val
        elif sample in ('GFP', 'MC'):
            records.setdefault(sample, {})[f'{sample}_initial'] = val

    for sample, val in ploidy_week5.items():
        if sample.startswith('C'):
            records.setdefault(sample, {})['Control Week 5'] = val
        elif sample.startswith('F'):
            records.setdefault(sample, {})['Fusion Week 5'] = val
        elif sample in ('GFP', 'MC'):
            records.setdefault(sample, {})[f'{sample}_week5'] = val

    summary_df = pd.DataFrame.from_dict(records, orient='index')

    col_order = ['Control Initial', 'Control Week 5',
                 'Fusion Initial', 'Fusion Week 5',
                 'GFP_initial', 'GFP_week5', 'MC_initial', 'MC_week5']
    col_order = [c for c in col_order if c in summary_df.columns]
    summary_df = summary_df[col_order]

    summary_df = summary_df.reindex(
        sorted(summary_df.index, key=lambda s: (s[0] not in 'CF', s))
    )

    return summary_df

# Main

In [ ]:
def load_g1_means(path):
    """Load a G1-only CSV (one row labeled 'Mean G1', one column per sample)."""
    df = pd.read_csv(path, index_col=0)
    return df.loc['Mean G1']

# Load raw G1 peak values
initial_raw = load_g1_means('/stor/work/Brock/kennedy/SC_repo/data/Ploidy/MDA-MB-231_ploidy/231_initial_g1only.csv')
week5_raw = load_g1_means('/stor/work/Brock/kennedy/SC_repo/data/Ploidy/MDA-MB-231_ploidy/231_5W_g1only.csv')

# --- Initial timepoint: single shared control (SFF_C) ---
control_initial = initial_raw['SFF_C']
ploidy_initial = (initial_raw.drop('SFF_C') / control_initial) * 3

# --- Week 5 timepoint: SFF_C is the control for everyone EXCEPT C8, 
#     which uses its own dedicated SFF_C_C8 control ---
control_week5 = week5_raw['SFF_C']
control_week5_c8 = week5_raw['SFF_C_C8']

week5_samples = week5_raw.drop(['SFF_C', 'SFF_C_C8'])
ploidy_week5 = pd.Series(index=week5_samples.index, dtype=float)

for sample, g1_val in week5_samples.items():
    control_val = control_week5_c8 if sample == 'C8' else control_week5
    ploidy_week5[sample] = (g1_val / control_val) * 3

print("Initial ploidy:\n", ploidy_initial)
print("\nWeek 5 ploidy:\n", ploidy_week5)

In [ ]:
ploidy_summary_231 = build_ploidy_summary_231(ploidy_initial, ploidy_week5)
ploidy_summary_231

print("\nPloidy Averages")
for column in ploidy_summary_231.columns:
    average = ploidy_summary_231[column].mean()*23
    standard_error = ploidy_summary_231[column].sem()
    standard_dev = ploidy_summary_231[column].std()
    print(f"\n{column}")
    print("Average:", average)
    print("Standard Error:", standard_error)
    print("Standard Deviation:", standard_dev,'\n')

In [ ]:
p_values = stats_and_plot(ploidy_summary_231.iloc[:, :-4], alpha=0.05, sig_bars=True, point_label=False, ylabel='MDA-MB-231 Ploidy\n(N = 23 chromosomes)', chromosomes_per_n=1, ploidy_axis=True, fig_size=(8, 6), title='MDA-MB-231 Average ploidy Week 0 vs 5', save_svg=True)
p_values